# b3_report

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): scoring and the gate. `poolScores` sums the experiments' score files; `gateB3` applies Gate B3's frozen counting rule (H22-1, P22-10: fitted right strictly above unfitted, pilot-only and every-design-helps on the pooled common set, at least 8 conclusive items), with an exact two-sided McNemar p beside each comparison, H22-4 (fitted against prior-only) and P22-17's pilot-computed comparison, and refuses while a frozen forecast has no score (FR-D3). `retroMarkdown`, `writeB3` and `figB3` write the week-1 and final reports (FR-D4); `syntheticB3` builds score files with known counts for the offline tests.

In [ ]:
#| default_exp b3_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from math import comb
from pathlib import Path

from explRL.b3_forecast import LINES, writeForecast
from explRL.b3_plan import B3_PLAN

ALTS = ('unfitted', 'pilotOnly', 'naive')                    # Gate B3's alternatives (H22-1)
FORECASTS = dict(C1='specs/multiagent/phase20/c1_predictions.json')   # C2 and C4 part b add theirs (P22-15)
P22_20 = ('In the unfitted simulator, relays feed scripted liars\' coupled solves (liars trust single posts), and honest coupled '
          'solves are 0 at θ_default (P20-17, P22-20).')
P22_16 = ('Pilot-only reads the pilot plus the Claude rows that existed before the experiment, including the reused reference '
          'cells; the simulators never see them. This favours pilot-only and makes Gate B3 harder for the fitted simulator (P22-16).')


def mcnemar(a, b):
    "Exact two-sided McNemar p of two forecasters' right/not-right lists over the same items (binomial on the discordant pairs)."
    n10, n01 = sum(x and not y for x, y in zip(a, b)), sum(y and not x for x, y in zip(a, b))
    n = n10 + n01
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, i) for i in range(min(n10, n01) + 1)) / 2 ** n)


def _cnt(items, lines=LINES):
    return {k: {o: sum(i['result'].get(k) == o for i in items) for o in ('right', 'wrong', 'no call')} for k in lines}


def poolScores(paths):
    "FR-D3: the items of every score file (scoreExperiment's), with counts on the common set and on all items, per experiment and pooled."
    items, byExp = [], {}
    for p in paths:
        s = json.loads(Path(p).read_text())
        byExp[s['exp']] = dict(common=s['common'], all=s['all'])
        items += [dict(exp=s['exp'], id=k, observed=v['observed'], represented=v['represented'], fallback=v['fallback'],
                       result=v['result']) for k, v in sorted(s['items'].items())]
    common = [i for i in items if i['represented']]
    comp = [i for i in items if 'pilotOnly' in i['result'] and not i['fallback']]
    return dict(experiments=sorted(byExp), byExp=byExp, items=items, common=_cnt(common), all=_cnt(items),
                pilotComputed=_cnt(comp, ('fitted', 'pilotOnly')), conclusive=sum(i['observed'] != '0' for i in common))


def forecastStatus(root='.', forecasts=FORECASTS):
    "{exp: dict(forecast, score)}: whether each experiment's frozen forecast and its score file exist."
    R = Path(root)
    return {e: dict(forecast=(R / p).exists(), score=(R / p.replace('_predictions', '_score')).exists()) for e, p in forecasts.items()}


def gateB3(pooled, plan=None, status=None):
    """FR-D3, H22-1 by P22-10's counting rule on the pooled common set; McNemar beside it; H22-4 descriptive. Refuses while an
    experiment of plan['experiments'] has a frozen forecast but no score; 'too few calls to read' below plan's minimum."""
    plan, status = plan or B3_PLAN, status or {}
    missing = [e for e, s in sorted(status.items()) if e in plan['experiments']['simulated'] and s['forecast'] and not s['score']]
    if missing:
        return dict(verdict='refused', missing=missing, note='every frozen forecast must be scored before the gate (FR-D3)')
    concl = [i for i in pooled['items'] if i['represented'] and i['observed'] != '0']
    right = {k: pooled['common'][k]['right'] for k in LINES}
    out = dict(conclusive=len(concl), minimum=plan['gate']['minConclusive'], right=right,
               wrong={k: pooled['common'][k]['wrong'] for k in LINES},
               mcnemar={a: mcnemar(*[[i['result'].get(k) == 'right' for i in concl] for k in ('fitted', a)]) for a in ALTS},
               H22_4=dict(fitted=right['fitted'], priorOnly=right['priorOnly'], note='descriptive'),
               pilotComputed=pooled['pilotComputed'])
    if len(concl) < plan['gate']['minConclusive']:
        return dict(out, verdict=plan['gate']['below'])
    return dict(out, verdict='passes' if all(right['fitted'] > right[a] for a in ALTS) else 'fails')

In [ ]:
#| export
def _table(counts, lines):
    head = '| Forecaster | Right | Wrong | No call |\n|---|---|---|---|\n'
    return head + ''.join(f"| {k} | {counts[k]['right']} | {counts[k]['wrong']} | {counts[k]['no call']} |\n" for k in lines if k in counts)


def retroMarkdown(r):
    "retro.md: per candidate, the counts on paper2's 11 conclusive calls, H22-2, Phase 14's calls, and the chosen knob's items."
    out = [f"# B3 retrospective check (week 1)\n\nChosen knob: `{r['knob']}`. {r['caveat']}.\n"]
    for k, s in r['summary'].items():
        out.append(f"\n## Candidate `{k}`{' (chosen)' if k == r['knob'] else ''}\n\n{s['conclusive']} conclusive calls. H22-2 "
                   f"(fitted right > the unfitted 9): {'holds' if s['H22_2'] else 'does not hold'}.\n\n" + _table(s['counts'], LINES))
        if s['P14']:
            out.append("\nPhase 14's two team-size calls (simulators only):\n\n" + _table(s['P14'], ('fitted', 'priorOnly', 'unfitted')))
        if s['unfittedMismatch']:
            out.append(f"\nThe unfitted line differs from the frozen signs on {s['unfittedMismatch']}.\n")
    out.append('\n## Items, chosen knob\n\n| Fold | Item | Haiku interval | ' + ' | '.join(LINES) + ' |\n|' + '---|' * (3 + len(LINES)) + '\n')
    for f, d in r['folds'][r['knob']].items():
        for i, v in d['items'].items():
            out.append(f"| {f} | {i} | [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}] | " +
                       ' | '.join(f"{v['lines'].get(k, '')} {v['calls'].get(k, '')}" for k in LINES) + ' |\n')
    out.append('\nThe error and collude faults and the plain-board liar cells are not fit (P22-3): rule agents take the newest value on '
               "the plain board (contamination 0.94-0.99 against Claude's 0.72-0.85), and no knob changes it.\n")
    return ''.join(out)


def writeB3(fitRes, retro, pooled, gate, out):
    "FR-D4: report.md and score.json in out (results/multiagent/b3)."
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    md = [f"# B3: a better cheap screen\n\n## Fit\n\nKnob `{fitRes['knob']}`; θ0 on all cells {fitRes['fits']['all']}; τ {fitRes['tau']}; "
          f"loss {fitRes['loss']['all']:.1f} against θ_default's {fitRes['loss']['default']:.1f}; fit health "
          f"{'ok' if fitRes['health']['ok'] else 'NOT ok'} (H22-3 per candidate {fitRes['health']['H22_3']}).\n",
          f"\n## Retrospective (H22-2)\n\nChosen knob: fitted right {retro['summary'][retro['knob']]['counts']['fitted']['right']} of "
          f"{retro['summary'][retro['knob']]['conclusive']} (unfitted 9, every-design-helps 10). {retro['caveat']}.\n",
          '\n## Forward forecasts\n']
    for e in pooled['experiments']:
        md.append(f"\n### {e}\n\nCommon set:\n\n{_table(pooled['byExp'][e]['common'], LINES)}\nAll items:\n\n"
                  f"{_table(pooled['byExp'][e]['all'], LINES)}" + (f'\n{P22_20}\n' if e == 'C1' else ''))
    md.append(f"\n### Pooled\n\nCommon set ({gate.get('conclusive', 0)} conclusive):\n\n{_table(pooled['common'], LINES)}\n"
              f"All items:\n\n{_table(pooled['all'], LINES)}\n{P22_16}\n")
    md.append(f"\n## Gate B3 (H22-1): {gate['verdict']}\n\n" + ('' if gate['verdict'] == 'refused' else
              f"McNemar p, fitted against each: {gate['mcnemar']}. H22-4 (descriptive): {gate['H22_4']}. Fitted against pilot-only on "
              f"the items pilot-only computes: {gate['pilotComputed']}.\n"))
    (out / 'report.md').write_text(''.join(md))
    writeForecast(dict(pooled=pooled, gate=gate, knob=fitRes['knob']), out / 'score.json')
    return ''.join(md)


def figB3(pooled, retro, path):
    "b3.png: right, wrong and no-call counts per forecaster, per experiment and pooled, with the retrospective beside."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    panels = [(e, pooled['byExp'][e]['common']) for e in pooled['experiments']] + [('Pooled', pooled['common']),
              ('Retrospective', retro['summary'][retro['knob']]['counts'])]
    fig, axs = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.2), sharey=True, squeeze=False)
    for ax, (t, c) in zip(axs[0], panels):
        bottom = [0] * len(LINES)
        for o, col in (('right', '#2a7d4f'), ('wrong', '#b8432f'), ('no call', '#b9b4aa')):
            v = [c[k][o] for k in LINES]
            ax.bar(range(len(LINES)), v, bottom=bottom, color=col, label=o, edgecolor='white', linewidth=1)
            bottom = [a + b for a, b in zip(bottom, v)]
        ax.set_xticks(range(len(LINES)), ['fitted', 'prior', 'unfitted', 'pilot', 'naive'], rotation=45, fontsize=8)
        ax.set_title(t, fontsize=10)
        ax.spines[['top', 'right']].set_visible(False)
    axs[0][0].set_ylabel('effects')
    axs[0][-1].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def syntheticB3(d):
    "Score files with known counts in d (two experiments, 10 conclusive represented items), and a fit and retrospective with known values."
    d, paths = Path(d), []
    for exp, n in (('C1', 6), ('C2', 6)):
        it = {f'{exp}-{i}': dict(observed='+' if i < 5 else '0', represented=i != 5, fallback=i == 4,
                                 result=dict(fitted='right' if i < 4 else 'wrong' if i < 5 else 'no call', priorOnly='right' if i < 3 else 'wrong',
                                             unfitted='right' if i < 2 else 'wrong', pilotOnly='right' if i < 3 else 'wrong',
                                             naive='right' if i < 3 else 'wrong')) for i in range(n)}
        s = dict(exp=exp, items=it, common=_cnt([v for v in it.values() if v['represented']]), all=_cnt(list(it.values())))
        writeForecast(s, d / f'{exp.lower()}_score.json')
        paths.append(d / f'{exp.lower()}_score.json')
    c = {k: dict(right=9 + (k == 'fitted'), wrong=2 - (k == 'fitted'), **{'no call': 4}) for k in LINES}
    fitRes = dict(knob='spread', fits=dict(all=dict(trustOne=0.4, spread=0.3)), tau=dict(trustOne=0.1, spread=0.1),
                  loss=dict(all=40.0, default=90.0), health=dict(ok=True, H22_3=dict(spread=True)))
    retro = dict(knob='spread', caveat='in-sample', folds=dict(spread={}),
                 summary=dict(spread=dict(conclusive=11, counts=c, H22_2=True, P14=None, unfittedMismatch=[])))
    return paths, fitRes, retro

## Tests

In [ ]:
# test_b3_files (AC-9, FR-D3, FR-D4): poolScores' counts, gateB3's rule, its minimum and its refusal, McNemar, writeB3 and figB3 on syntheticB3
import tempfile
d = Path(tempfile.mkdtemp())
paths, fitRes, retro = syntheticB3(d)
P = poolScores(paths)
assert P['experiments'] == ['C1', 'C2'] and P['conclusive'] == 10 and len(P['items']) == 12
assert P['common']['fitted'] == {'right': 8, 'wrong': 2, 'no call': 0} and P['common']['unfitted']['right'] == 4
assert P['all']['fitted']['no call'] == 2 and P['pilotComputed']['fitted']['right'] == 8   # the fallback items (i = 4) left out
G = gateB3(P)
assert G['verdict'] == 'passes' and G['right']['fitted'] == 8 and G['right']['naive'] == 6 and G['conclusive'] == 10
assert G['mcnemar']['unfitted'] == mcnemar([1] * 8 + [0] * 2, [1] * 4 + [0] * 6) == 0.125
assert gateB3(P, status=dict(C1=dict(forecast=True, score=False)))['verdict'] == 'refused'
assert gateB3(P, status=dict(C1=dict(forecast=True, score=True)))['verdict'] == 'passes'
few = dict(P, items=[i for i in P['items'] if i['exp'] == 'C1'])
assert gateB3(few)['verdict'] == 'too few calls to read'
tie = dict(P, common={**P['common'], 'naive': dict(P['common']['fitted'])})
assert gateB3(tie)['verdict'] == 'fails'                                   # a tie fails (strictly above)
assert mcnemar([1, 1], [1, 1]) == 1.0 and mcnemar([1] * 5, [0] * 5) == 0.0625
md = writeB3(fitRes, retro, P, G, d)
assert 'Gate B3 (H22-1): passes' in md and 'P22-20' in md and 'P22-16' in md and (d / 'score.json').exists()
figB3(P, retro, d / 'b3.png')
assert (d / 'b3.png').stat().st_size > 10000
assert 'Candidate `spread` (chosen)' in retroMarkdown(retro)
assert forecastStatus('/nonexistent') == dict(C1=dict(forecast=False, score=False))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()